In [2]:
import torch
import torch.nn as nn
import torch.utils.data as data
import math
import numpy as np

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda_is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"设备: {device}")


设备: mps


我的个人电脑是M1系列的Mac电脑，上面的执行结果是：设备: mps，如果你是用有NVIDIA GPU的电脑，执行结果应该是：设备: cuda，手头没有amd的gpu，不太清楚运行结果是啥。如果读者有amd的gpu，可以试下是cuda还是cpu，应该是有amd专门的pytorch版本的。


# 1. 构建迷你翻译数据集和词表
为了演示就先不用完整的数据集了，我们创建一个微型的“Demo数据集”，看清token是如何被处理的。你就知道为什么大模型很多都是用Token这种奇奇怪怪的单位来计费了，因为Token才是大模型的计算原子。

In [ ]:
# 构建原始数据集，source 是英文，target 是中文
raw_data = [
    ("I love deep learning", "我爱深度学习"),
    ("Transformer is all you need", "你需要的就是变形金刚"),
    ("Attention mechanism is great", "注意力机制很棒"),
    ("hello world", "你好世界"),
    ("PyTorch is easy to learn", "PyTorch很容易学")
]

# 构建词表，为了简单起见，英文用空格，中文用单个字符，在实际项目汇总可以用BPE（Byte Pair Encoding）等方法构建更复杂的词表。
# 这里pad sos eos分别代表填充、开始、结束
src_vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
tar_vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2}

# 扫描数据集，构建词表
for src, tar in raw_data:
    for word in src.split():
        if word not in src_vocab:
            src_vocab[word] = len(src_vocab)
    for word in tar:
        if word not in tar_vocab:
            tar_vocab[word] = len(tar_vocab)

# 创建反向词表（索引 -> 单词）
idx2src = {v: k for k, v in src_vocab.items()}
idx2tar = {v: k for k, v in tar_vocab.items()}

print(f"源语言词表: {src_vocab}")
print(f"目标语言词表: {tar_vocab}")


上面我们已经完成了词表构建，如果是想训练构建尽可能完整的中英文的词表，那我们就要搜索海量的文本数据，比如百度百科，维基百科，全部爬下来，然后通过tokenizer将文本转换为token，再构建词表。

给大家一个数字，让大家感受下。
英文日常交流使用 3000-4000个单词就可以覆盖 90%的口语场景，在中文领域，常用的汉字大概在 3000-5000个汉字之间。



| 维度            | 英文 (English)               | 中文 (Chinese)                                                   |
| ------------- | -------------------------- | -------------------------------------------------------------- |
| 人类常用基础单元      | ~20,000 单词 (大学水平)          | ~3,500 汉字 (常用字)                                                |
| 人类常用词汇/短语     | ~50,000 (韦氏基础词)            | ~56,000 常用词 (现代汉语词表)                                           |

GPT3使用的词表大概是 50k（数据主要是英文），Qwen3使用的词表已经是 150K（中英文都支持）了，GPT4使用的词表是100k，GPT4o使用的词表是200K，可以看到词表一直在扩大，也就代表训练的数据量也在增加。

# 2. Dataset与DataLoader （Padding）
很多初学者像我一样有疑问，怎么把长短不一的句子塞到同一个Batch中呢？
答案，反正只是为了长度一样，我就随便塞点无意义的0进去就好了。

In [ ]:
class ToyTranslationDataset(data.Dataset):
    # 初始化函数，将数据集（格式如上，是一个元组的列表），源语言词表，目标语言词表传进来。
    def __init__(self, data, src_vocab, tar_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.tar_vocab = tar_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text, tar_text = self.data[idx]
        # 这里是遍历的语法糖，查找原始数据的每个token在词表中对应的位置，将token转化为数字格式
        src_indices = [self.src_vocab[word] for word in src_text.split()]  # 英文按空格分词
        tar_indices = [self.tar_vocab[word] for word in tar_text]  # 中文按单个字符遍历

        return torch.tensor(src_indices), torch.tensor(tar_indices)

 神经网络要求输入是规整的矩阵，不能是长度参差不齐的列表，对于文本中如果长短不一致的话，无法训练怎么办呢，就要用下面的补0策略，将对应的数据补好0，如果你有cnn的经验那大概率知道这是在干嘛，如果不知道的话，忽略就好啦，知道这里是为了后面训练保持一致填充了一些无意义信息就好。

In [ ]:
def collate_fn(batch):
    """
    自定义的batch处理函数，batch是一个列表，包含__getitem__ 返回的元组，这个函数就是用来进行补前后标识，补0的。

    :param batch:
    :return:
    """
    src_batch, tar_batch = [], []
    #
    for src_sample, tar_sample in batch:
        # 源序列就保持不变，简单处理
        src_batch.append(src_sample)
        # 目标序列拼接开头sos，结尾拼上eos，让transformer知道从哪里开始到哪里结束
        tar_processed = torch.cat([
            torch.tensor([tar_vocab['<sos>']]),
            tar_sample,
            torch.tensor([tar_vocab['<eos>']])
        ])
        tar_batch.append(tar_processed)

    # padding_value=0 是我们在词表中定义的 <pad>
    src_batch = nn.utils.rnn.pad_sequence(src_batch, padding_value=0, batch_first=True)
    tar_batch = nn.utils.rnn.pad_sequence(tar_batch, padding_value=0, batch_first=True)
    return src_batch, tar_batch


dataset = ToyTranslationDataset(raw_data, src_vocab, tar_vocab)
# 这里的参数，就是将dataset中的数据，按照 batch_size（当前为2）为一批，shuffle就是不打乱，避免每次输出结果不一致，使用刚才我们写的collate_fn处理
loader = data.DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

src_batch, tar_batch = next(iter(loader))

print("先看下没有补0的原始数据形状，是不是不一样长，可以自己数下，最长的有几个")
print("source的数据", dataset[0][0], dataset[1][0])
print("target数据", dataset[0][1], dataset[1][1])

print("\n处理后Batch数据的形状")
print(f"Source Batch Shape: {src_batch.shape} (Batch, Src_len) ")
print(f"Target Batch Shape: {tar_batch.shape} (Batch, Tar_len) ")
print("\n处理后Batch数据内容")
print("Source:\n", src_batch)
print("Target:\n", tar_batch)

讲解：上面batch等于2，含义就是用两组数据（中英文一起算一组）为一批去训练transformer，源数据（英文）一个句长度为4，一个长度为5，则将短的句子最后一位补0。
目标数据因为要加 sos 和 eos，你会发现最长的句子长度比原来多了2，其他句子也被补0，补到了12。

# 3. 词嵌入（Token Embedding)

经过dataLoader，我们已经把人类的语言初步转换成了数字，也就是每个Token对应的一个词表中的Index，想一下我们是否可以直接用这些Index去训练了呢？反正电脑是能看懂数字的，让他们去学就好了呗，类比一下这就像老师给你布置了一本寒假作业，只给了你目录，让你把整本作业连题目带答案都写出来。

所以我们要给大模型更充分的信息，将这个词扩展到更高维度，让这个词在现实生活中的众多含义都能被数字表示，比如香蕉的含义有水果，黄色，热带，弯的等等，你只用一个数字编号很难表征这些含义。
| 整数索引 | Embedding 向量 |
|---|---|
| 1 维，只有"编号"信息 | 768 维，每个维度都可以编码不同的语义特征 |
| 无法表达词与词之间的相似度 | 向量距离天然反映语义距离（如 `king - man + woman ≈ queen`） |
| 不同 token 之间没有可比性 | 语义相近的词在向量空间中聚集 |

好，那现在我们就让小token升维到高维生物，让他能更完整的表示自己。
nn.Embedding(vocab_size, emb_size) 本质上就是一个形状为（vocab_size, emb_size），vocab_size就是你的词表大小，你的词表是5000个token，这里就是5000，emb_size其实就是d_model，是要将这个向量映射到多少维的高维空间，为什么要有两个变量描述同一个值呢，因为描述的视角不同，emb_size是站在Embedding层设计的，d_model是站在整个模型架构层界面设计，是在模型中流动的数据的宽度。

In [8]:
# 我们创建一个小的Embedding，假设词表只有10，向量维度为4.
emb_layer = nn.Embedding(num_embeddings=10, embedding_dim=4)

print("Embedding 权重矩阵形状：", emb_layer.weight.shape)
print("Embedding 权重矩阵值：", emb_layer.weight.data)

# 假设一个句子是通过词表的第3，0，7个token组成
input_ids = torch.tensor([3, 0, 7])
outputs = emb_layer(input_ids)

print(f"\n输入索引: {input_ids}")
print(f"输出形状: {outputs.shape}")
print(f"输出向量: {outputs.data}")

# 验证下Embedding本质就是查表操作，没做任何计算
print(f"\n验证 - Embedding层第三行: {emb_layer.weight.data[3]}")
print(f"验证 -Outputs {outputs[0]}")
print(f"两者相等：{torch.equal(emb_layer.weight.data[3], outputs.data[0])}")

# 模拟batch输入 （batch = 2, seq_len = 3)
batch_input = torch.tensor([[1, 2, 3], [4, 5, 0]])
batch_output = emb_layer(batch_input)
print(f"\nBatch 输入形状：{batch_input.shape}")
print(f"Batch输出形状：{batch_output.shape}")
print(f"Batch输出数据：{batch_output.data}")

# 论文中还有一个Embedding * sqrt(d_model)， 这里就是为了将语义信息给予大权重，避免被位置信息“功高盖主”，比如你喝一杯咖啡，加了一杯糖浆，你还能喝出来啥，你要是加一大杯糖，你是不是得用一桶咖啡，同理
d_model = 512
emb_large = nn.Embedding(100, d_model)
sample = emb_large(torch.tensor([1]))
print(f"\n未缩放的 Embedding 均值：{sample.mean().item():.4f} 标准差：{sample.std().item():.4f}")
sample_scaled = sample * math.sqrt(d_model)
print(f"\n缩放后的 Embedding 均值：{sample_scaled.mean().item():.4f} 标准差：{sample_scaled.std().item():.4f}")
print(f"缩放因子 sqrt({d_model}) = {math.sqrt(d_model):.2f}")


Embedding 权重矩阵形状： torch.Size([10, 4])
Embedding 权重矩阵值： tensor([[ 1.9020,  0.6483,  1.0186,  0.9851],
        [ 0.5901, -0.6783,  0.7546, -1.1118],
        [-1.2893,  0.2385, -0.8145,  0.1143],
        [ 0.1614,  0.7830, -0.2108, -0.9183],
        [ 0.1901, -0.5253,  0.8648, -0.2805],
        [ 0.1915,  0.3454, -0.5697,  1.6607],
        [-0.4164, -0.7904, -0.5354, -0.9109],
        [ 0.1154,  0.4978, -0.2712,  1.4515],
        [ 0.6374, -2.1763,  0.4765, -0.3528],
        [-0.5338,  0.1227,  3.4582,  2.0376]])

输入索引: tensor([3, 0, 7])
输出形状: torch.Size([3, 4])
输出向量: tensor([[ 0.1614,  0.7830, -0.2108, -0.9183],
        [ 1.9020,  0.6483,  1.0186,  0.9851],
        [ 0.1154,  0.4978, -0.2712,  1.4515]])

验证 - Embedding层第三行: tensor([ 0.1614,  0.7830, -0.2108, -0.9183])
验证 -Outputs tensor([ 0.1614,  0.7830, -0.2108, -0.9183], grad_fn=<SelectBackward0>)
两者相等：True

Batch 输入形状：torch.Size([2, 3])
Batch输出形状：torch.Size([2, 3, 4])
Batch输出数据：tensor([[[ 0.5901, -0.6783,  0.7546, -1.1118],
         

# 4. 位置编码
好，你已经解决了一个大问题，怎么把人类看的语言，彻底转化成了电脑能看懂的语言（向量），但是给电脑的信息是不是不够全，我们还没告诉他这些数字的顺序，好，现在你就可以反驳我了，我提供的数据不就是按顺序排列的吗，怎么能说我没给电脑顺序呢。它一个个读不就知道顺序了吗，非常好的问题，如果是串行读的话，模型确实可以根据读入的顺序推断，如果你耳闻过RNN，LSTM等模型的话，你就知道有多慢，transformer是一次性并行处理所有token的，所以没有位置编码的话，它看到I love you, Love I you是一样的。所以我们需要在上一步生成的表义数字中加上一个位数相同的“位置向量”。
最原始的公式， (不用死记，知道原理即可)：
原理有一个B站视频讲的非常好，只有10分钟，有兴趣的可以看下 https://b23.tv/qpalhST，  我个人理解的一句话解释就是，基于sin和cos的周期性质和函数的上下限，通过叠加不同尺度的sin和cos来对位置进行编码，值范围控制在固定范围方便归一化，并且任意位置的pos+K编码都能通过pos编码通过线性函数获得。

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$

$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # 创建一个（max_len, d_model) 大小的矩阵来存位置编码，
        # max_len 不是刚才计算的最长序列的长度，而是一个预先分配的上限，相当于九九乘法表，后续可以直接查就行了，如果有训练数据长度超过5000，可以动态计算，也可以初始把这个值发大一点。
        # d_model
        pe = torch.zeros(max_len, d_model)

        # 生成位置索引，并在最后加一维度: [0, 1, 2, ... max_len-1] -> (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 计算分母中的 div_term
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # 偶数下标用sin
        pe[:, 0::2] = torch.sin(position * div_term)
        # 奇数下标用cos
        pe[:, 1::2] = torch.cos(position * div_term)

        # 增加一个维度，变成(1, max_len, d_model)，才能和后续的batch数据相加
        pe = pe.unsqueeze(0)

        # register_buffer 表明这是不需要学习的参数，随查随用
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

我们来一起推导下，最终的位置矩阵长什么样，先简化一下，max_len = 4，d_model = 8的场景

position = [[0,1,2,3]]

div_term = [1.0, 0.1, 0.01, 0.001] 这个可以自行计算下 10000的0次方，1/4次方，1/2次方，3/4次方的倒数，是不是就是这些书。

position × div_term 就是一个(4, 4)的矩阵

```
pos=0: [0.0,  0.0,  0.0,   0.0  ]
pos=1: [1.0,  0.1,  0.01,  0.001]
pos=2: [2.0,  0.2,  0.02,  0.002]
pos=3: [3.0,  0.3,  0.03,  0.003]
```

然后再算下每个位置的sin和cos对应的值，得到下面的对照表

```
sin(pos*div):                         cos(pos*div):
pos=0: [ 0.0,    0.0,   0.0, 0.0]    pos=0: [ 1.0,    1.0,   1.0, 1.0]
pos=1: [ 0.84,   0.10,  0.01, 0.0]   pos=1: [ 0.54,   0.99,  1.0, 1.0]
pos=2: [ 0.91,   0.20,  0.02, 0.0]   pos=2: [-0.42,   0.98,  1.0, 1.0]
pos=3: [ 0.14,   0.30,  0.03, 0.0]   pos=3: [-0.99,   0.96,  1.0, 1.0]
```

还记得我们上面说的，交错拼接吗，偶数用sin，奇数用cos，计算机是从0开始的下标。

拼接之后就是

```
列:     sin₀   cos₀   sin₁   cos₁   sin₂   cos₂   sin₃   cos₃
pos=0: [ 0.0,   1.0,   0.0,   1.0,   0.0,   1.0,   0.0,   1.0 ]
pos=1: [ 0.84,  0.54,  0.10,  0.99,  0.01,  1.0,   0.001, 1.0 ]
pos=2: [ 0.91, -0.42,  0.20,  0.98,  0.02,  1.0,   0.002, 1.0 ]
pos=3: [ 0.14, -0.99,  0.30,  0.96,  0.03,  1.0,   0.003, 1.0 ]
```


能看到左边的列因为sin和cos的周期小，频率高，不同位置之间差异比较明显，而右边的列则相反，不同位置之间的差异则较小，且能推理出来，后面会越来越小。
就像一个时钟，左边是秒针，右边是时针，中间还有各种分针，通过不同频率就能组合出来一个唯一的位置编码指纹。

既然我都类比时钟了，聪明的你会有疑问，时钟上的时间是周期性的呀，会重置的，那会不会有位置编码因为长度太长了而一样呢？
最简单一层理解：单个维度是会周期重复的，但是这么多不同频率组合在一起，你可以想象下我们有年月日时分秒，更久远的有decade，世纪，纪元等等。
第二层理解：所有维度同时回到相同值才会一样，那么也就是所有维度周期的最小公倍数，周期是无理数，不会有这个问题。
第三层理解：聪明的你知道计算机是有精度限制的，不会存在真的无理数，这里我只能告诉你，对于咱们这个小模型够用啦，现在新的大模型有别的位置编码方案。

然后我们再来推导一下，每一批数据经过这一层的forward都会发生什么变化。
假设 x 的 shape是 （32, 50, 512）(batch = 32, sel_len = 50, d_model = 512)
```
self.pe 的 shape （1, 5000, 512)
self.pe 截取 [:, :50, :] -> (1, 50, 512)
x + self.pe[:, :50, :] -> (32, 50, 512) 会广播到每个样本上，
```
即batch 维度 1广播到32， 每个样本加同样的位置编码，（这也很朴素的认知，位置和句子的具体内容无关对吧，你爱我，我爱你，蜜雪冰城甜蜜蜜，你，我，蜜，这三个字都是句首的位置，理应用index = 0的位置编码。

# 组合 Embedding + Positional Encoding



好，现在我们来把Embedding层和位置编码层整合一下，这一步就是将上面的代码封装为一个类，很简单。

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)


class TransformerInputLayer(nn.Module):
    def __init__(self, vocab_size, emb_size, max_len=5000, dropout=0.1):
        super().__init__()
        self.token_emb = TokenEmbedding(vocab_size, emb_size)
        self.pos_emb = PositionalEncoding(emb_size, max_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = self.token_emb(x)
        out = self.pos_emb(out)
        return self.dropout(out)


至此，我们已经准备好了Transformer的原材料，只要逐行写完了上面的代码，运行没有报错，我们就可以进入第二部分了，手写注意力机制。